In [2]:
%pip install -q ultralytics==8.4.1 pycocotools==2.0.11 tqdm
!git clone https://github.com/mdmohsin212/WasteScope.git

Cloning into 'WasteScope'...
remote: Enumerating objects: 522, done.
remote: Counting objects: 100% (522/522), done.
remote: Compressing objects: 100% (291/291), done.
remote: Total 522 (delta 287), reused 426 (delta 197), pack-reused 0 (from 0)
Receiving objects: 100% (522/522), 28.86 MiB | 21.89 MiB/s, done.
Resolving deltas: 100% (287/287), done.


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import hashlib
import importlib.util
import json
import math
import platform
import shutil
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from tqdm.auto import tqdm
from ultralytics import YOLO

In [5]:
BASE = Path("/content/drive/MyDrive/WasteScope")
RUN_NAME = "S2_yolo26n_seg_roi_960"

IMGSZ, ROI_BATCH = 960, 8
CONF, AP_CONF, MAX_DET = 0.20, 0.001, 100

WORK = Path("/content/wastescope_finalize") / RUN_NAME
TRAIN_RUN = BASE / "runs/segment" / RUN_NAME
REPORT = BASE / "reports/segmentation" / RUN_NAME / "test_best_960_fused_conf0.2"

DATA = WORK / "data"
WORK.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

In [6]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def environment():
    return {"python": platform.python_version(), "torch": torch.__version__,
            "ultralytics": version("ultralytics"), "pycocotools": version("pycocotools"),
            "numpy": np.__version__, "opencv": cv2.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

print("Reports:", REPORT)

Reports: /content/drive/MyDrive/WasteScope/reports/segmentation/S2_yolo26n_seg_roi_960/test_best_960_fused_conf0.2


In [ ]:
policy = json.loads((TRAIN_RUN / "roi_policy.json").read_text())

checkpoint_sources = {
    "detector": BASE / "runs/detect/D1_yolo26n_960/weights/best.pt",
    "full_image": BASE / "runs/segment/S1_yolo26n_seg_960/weights/best.pt",
    "roi": TRAIN_RUN / "weights/best.pt"
}

expected_hashes = {
    "detector": "cd6b987150bb2106321e59b61ac080a043d451994eb1e1cf99bd192a271109ea",
    "full_image": "1b8b41a691c0e51691565d4a50ffe7db50aaab144050b62b040548eae547a670",
    "roi": "c819c5b1e82228f33fd86e72365f8f841898acd88de2411f9e4541684256356f"
}

weights = {}
for branch, source in checkpoint_sources.items():
    target = WORK / f"{branch}.pt"
    shutil.copy2(source, target)

    if sha256(target) != expected_hashes[branch]:
        raise ValueError(f"{branch}: checkpoint differs from the validation-selected weights.")

    weights[branch] = target
    
REPO_DIR = Path("/content/WasteScope")
helper_path = REPO_DIR / "scripts/segmentation/s2_monitor.py"
evaluator_path = REPO_DIR / "src/evaluation/metrics.py"

mask_pipeline = load_module("frozen_s2_masks", helper_path)
canonical = load_module("frozen_segmentation_metrics", evaluator_path)

In [8]:
manifest_source = TRAIN_RUN / "video_split.json"
selection = {
    "system": "S2_fused",
    "run": RUN_NAME,
    "selected_on": "val",
    "result_stage": "final",
    "operating_confidence": CONF,
    "confidence_rule": ">",
    "imgsz": IMGSZ,
    "selection_reason": (
        "Best canonical validation mask AP50-95; confidence 0.20 "
        "reduces false alarms with near-best validation F1."
    ),
    "validation": {
        "mask_AP50": 0.6649369397204824,
        "mask_mAP50_95": 0.3650212107598986,
        "confidence": 0.20,
        "TP": 471,
        "FP": 144,
        "FN": 390,
        "precision": 471 / 615,
        "recall": 471 / 861,
        "F1": 942 / 1476,
        "background_false_alarm_images": 84,
        "background_images": 401,
    },
    "checkpoints": {
        key: {
            "path": str(path.relative_to(BASE)),
            "sha256": expected_hashes[key],
        }
        for key, path in checkpoint_sources.items()
    },
    "roi_best_epoch": 1,
    "roi_policy": policy,
    "test_proposals": ("D1 predictions only; no GT proposals, jitter " "or full-frame ROI fallback"),
    "inference": {
        "AP_confidence_min": AP_CONF,
        "max_detections": MAX_DET,
        "detector_max_det": 300,
        "half": True,
        "rect": False,
        "retina_masks": True,
        "roi_batch": ROI_BATCH,
        "fusion": (
            "score-ordered mask-IoU NMS >= 0.5; "
            "S1 first on score ties"
        ),
    },
    "manifest_sha256": sha256(manifest_source),
    "evaluator_source_sha256": sha256(evaluator_path),
    "fusion_helper_sha256": sha256(helper_path),
    "versions": {
        "ultralytics": "8.4.1",
        "pycocotools": "2.0.11",
    },
    "checkpoint_selection_limitation": (
        "S0/S1 used native selection; S2 used canonical fused AP. "
        "Not a controlled architecture-only comparison."
    ),
}

selection_id = hashlib.sha256(json.dumps(selection, sort_keys=True).encode()).hexdigest()
frozen_path = REPORT / "selected_system.json"

if frozen_path.is_file():
    previous = json.loads(frozen_path.read_text())
    if previous != selection:
        raise ValueError(
            "This report already has a different frozen selection. " "Use a new report folder."
        )

write_json(frozen_path, selection)
for source in (helper_path, evaluator_path, manifest_source, TRAIN_RUN / "roi_policy.json"):
    shutil.copy2(source, REPORT / source.name)

write_json(REPORT / "evaluation_config.json", {"status": "partial", "selection_id": selection_id})
print("Frozen: S2 fused | confidence >", CONF, "| checkpoint epoch 1")

Frozen: S2 fused | confidence > 0.2 | checkpoint epoch 1


In [ ]:
annotations_path = DATA / "test/annotations.json"
ready = DATA / ".test_extracted"

if not ready.is_file():
    archive_path = WORK / "test.zip"
    
    if not archive_path.is_file():
        print("Copying prepared segmentation test.zip locally…", flush=True)
        temporary = archive_path.with_suffix(".zip.part")
        shutil.copy2(BASE / "prepared/segmentation/test.zip", temporary)
        temporary.replace(archive_path)
        
    with ZipFile(archive_path) as archive:
        members = [item for item in archive.infolist() if not item.is_dir() and
                   (item.filename == "test/annotations.json" or item.filename.startswith("test/images/"))]
        
        for item in tqdm(members, desc="Extracting test"):
            archive.extract(item, DATA)
            
    ready.touch()
    archive_path.unlink()
    

coco = COCO(str(annotations_path))
images = coco.loadImgs(sorted(coco.getImgIds()))

manifest = json.loads(manifest_source.read_text(encoding="utf-8-sig"))
test_videos = {str(value) for value in manifest["splits"]["test"]}

cache_key = {"selection_id": selection_id, "test_annotations_sha256": sha256(annotations_path)}
PREDICTIONS = REPORT / "predictions.json"
CACHE_METADATA = REPORT / "prediction_metadata.json"

reuse_predictions = PREDICTIONS.is_file() and CACHE_METADATA.is_file()
shutil.copy2(annotations_path, REPORT / "coco_ground_truth.json")

print(f"Test: {len(images):,} images | {len(coco.anns):,} instances")
print("Predictions:", "reuse saved Drive predictions" if reuse_predictions else "generate on GPU")

Copying prepared segmentation test.zip locally…


Extracting test:   0%|          | 0/1111 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Test: 1,110 images | 846 instances
Predictions: generate on GPU


In [10]:
def expanded_rect(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    side = max(policy["crop_factor"] * (x2 - x1),
               policy["crop_factor"] * (y2 - y1), policy["minimum_side"])
    crop_w, crop_h = min(side, width), min(side, height)
    
    left = min(max((x1 + x2 - crop_w) / 2, 0), width - crop_w)
    top = min(max((y1 + y2 - crop_h) / 2, 0), height - crop_h)
    
    return math.floor(left), math.floor(top), math.ceil(left + crop_w), math.ceil(top + crop_h)


def add_masks(result, rect, image_id, image_shape):
    if result.masks is None:
        return []
    masks = result.masks.data.byte().cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    records = []
    
    for mask, score in zip(masks, scores):
        rle = mask_pipeline.project_mask(mask, rect, image_shape)
        if rle is not None:
            records.append(mask_pipeline.candidate(image_id, rle, score))
            
    return records


def predict_system(image, image_id):
    height, width = image.shape[:2]
    common = {"imgsz": IMGSZ, "device": 0, "half": True, "rect": False, "save": False, "verbose": False}
    
    proposals = detector.predict(image, conf=policy["detector_confidence"], max_det=300, **common)[0]
    full = full_model.predict(image, conf=AP_CONF, max_det=MAX_DET, retina_masks=True, classes=[0], **common)[0]
    
    records = add_masks(full, (0, 0, width, height), image_id, (height, width))
    rects = list(dict.fromkeys(expanded_rect(box, width, height) for box in proposals.boxes.xyxy.cpu().numpy()))
    
    crops = []
    for left, top, right, bottom in rects:
        crop = image[top:bottom, left:right]
        scale = min(1.0, policy["stored_long_side"] / max(crop.shape[:2]))
        if scale < 1:
            crop = cv2.resize(crop, (round(crop.shape[1] * scale), round(crop.shape[0] * scale)),
                              interpolation=cv2.INTER_LINEAR)
        crops.append(crop)
        
    for start in range(0, len(crops), ROI_BATCH):
        results = roi_model.predict(crops[start:start + ROI_BATCH], batch=ROI_BATCH, conf=AP_CONF, max_det=MAX_DET, retina_masks=True, classes=[0], **common)
        
        for result, rect in zip(results, rects[start:start + ROI_BATCH]):
            records.extend(add_masks(result, rect, image_id, (height, width)))
            
    fused = mask_pipeline.fuse_masks(records, policy["fusion_iou"], MAX_DET)
    return mask_pipeline.coco_records(fused, json_safe=True), len(rects)

In [11]:
if reuse_predictions:
    predictions = json.loads(PREDICTIONS.read_text())
    prediction_metadata = json.loads(CACHE_METADATA.read_text())
    
else:
    detector, full_model, roi_model = [YOLO(str(weights[key])) for key in ("detector", "full_image", "roi")]
    predictions, timing_rows = [], []
    
    for info in tqdm(images, desc="Test: D1 + S1 + S2"):
        path = DATA / "test/images" / Path(info["file_name"]).name
        image = cv2.imread(str(path))
        
        if image is None:
            raise FileNotFoundError(f"Cannot decode test image: {path}")
        torch.cuda.synchronize()
        started = time.perf_counter()
        
        records, roi_count = predict_system(image, info["id"])
        torch.cuda.synchronize()
        
        predictions.extend(records)
        timing_rows.append({"Image ID": info["id"], "ROIs": roi_count, "APCollectionMs": 1000 * (time.perf_counter() - started)})
        
    write_json(PREDICTIONS, predictions)
    pd.DataFrame(timing_rows).to_csv(REPORT / "ap_collection_times.csv", index=False)
    
    prediction_metadata = {"cache_key": cache_key, "environment": environment(),
                           "created_utc": datetime.now(timezone.utc).isoformat(),
                           "timing_scope": "AP collection at confidence 0.001, including first-call warmup; not deployment latency"}
    write_json(CACHE_METADATA, prediction_metadata)
    del detector, full_model, roi_model
    torch.cuda.empty_cache()
    
print(f"Saved/reused {len(predictions):,} fused mask predictions")

Test: D1 + S1 + S2:   0%|          | 0/1110 [00:00<?, ?it/s]

Saved/reused 102,157 fused mask predictions


In [12]:
evaluation = canonical.evaluate_coco(coco.dataset, predictions, iou_type="segm")
ap_table = canonical.coco_results(evaluation)
errors = canonical.detection_errors(evaluation, confidence=CONF, iou=0.50)
gt, dt = evaluation.cocoGt, evaluation.cocoDt

def divide(a, b):
    return float(a / b) if b else None

def union_mask(rles, height, width):
    return (mask_utils.decode(mask_utils.merge(rles)).astype(bool) if rles
            else np.zeros((height, width), dtype=bool))

def boundary(mask):
    thickness = max(1, round(0.02 * np.hypot(*mask.shape)))
    padded = cv2.copyMakeBorder(mask.astype(np.uint8), 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    eroded = cv2.erode(padded, np.ones((3, 3), np.uint8), iterations=thickness)[1:-1, 1:-1]
    return mask & ~eroded.astype(bool)

operating = defaultdict(list)
for pred in predictions:
    if pred["score"] > CONF:
        operating[pred["image_id"]].append(pred["segmentation"])
        
        
pixel_rows = []
for info in tqdm(images, desc="Pixel diagnostics"):
    image_id, height, width = info["id"], info["height"], info["width"]
    truth = union_mask([gt.annToRLE(ann) for ann in gt.imgToAnns[image_id]], height, width)
    
    predicted = union_mask(operating[image_id], height, width)
    intersection, truth_area, pred_area = int((truth & predicted).sum()), int(truth.sum()), int(predicted.sum())
    
    pixel_rows.append({"Image ID": image_id, "IntersectionPixels": intersection,
                       "GTPixels": truth_area, "PredPixels": pred_area,
                       "FPAreaPct": 100 * int((predicted & ~truth).sum()) / (height * width),
                       "IoU": divide(intersection, truth_area + pred_area - intersection),
                       "Dice": divide(2 * intersection, truth_area + pred_area)})
    
per_image = errors.per_image.join(pd.DataFrame(pixel_rows).set_index("Image ID"))
per_image["Video"] = per_image["Image"].map(lambda name: Path(name).stem.split("_", 1)[0])

creating index...
index created!
Loading and preparing results...
DONE (t=2.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=6.27s).
Accumulating evaluation results...
DONE (t=0.95s).


Pixel diagnostics:   0%|          | 0/1110 [00:00<?, ?it/s]

In [13]:
boundary_rows = []
threshold_index = int(np.flatnonzero(np.isclose(evaluation.params.iouThrs, 0.5))[0])
all_area = evaluation.params.areaRng[evaluation.params.areaRngLbl.index("all")]

for item in evaluation.evalImgs:
    if item is None or item["aRng"] != all_area:
        continue
    
    for pred_id, gt_id in zip(item["dtIds"], item["dtMatches"][threshold_index]):
        if pred_id in errors.reviewed_prediction_ids and gt_id:
            a = boundary(gt.annToMask(gt.anns[int(gt_id)]).astype(bool))
            b = boundary(dt.annToMask(dt.anns[pred_id]).astype(bool))
            
            boundary_rows.append({"Image ID": item["image_id"], "Prediction ID": int(pred_id), "GT ID": int(gt_id), "BoundaryIoU": divide(int((a & b).sum()), int((a | b).sum()))})
            
boundary_matches = pd.DataFrame(boundary_rows, columns=["Image ID", "Prediction ID", "GT ID", "BoundaryIoU"])
tp, fp, fn = map(len, (errors.matched_gt_ids, errors.false_positive_ids, errors.missed_gt_ids))

background = per_image[per_image["Objects"].eq(0)]
intersection = int(per_image["IntersectionPixels"].sum())
gt_pixels, pred_pixels = int(per_image["GTPixels"].sum()), int(per_image["PredPixels"].sum())

metrics = {
    "mask_AP50": float(ap_table.loc["All Classes", "AP@0.50"]),
    "mask_mAP50_95": float(ap_table.loc["All Classes", "AP@[.50:.95]"]),
    "mask_AR50_95": float(ap_table.loc["All Classes", "AR@[.50:.95]"]),
    "TP": tp, "FP": fp, "FN": fn, "precision": divide(tp, tp + fp),
    "recall": divide(tp, tp + fn), "F1": divide(2 * tp, 2 * tp + fp + fn),
    "overall_foreground_IoU": divide(intersection, gt_pixels + pred_pixels - intersection),
    "Dice": divide(2 * intersection, gt_pixels + pred_pixels),
    "boundary_IoU_matched": float(boundary_matches["BoundaryIoU"].mean()) if len(boundary_matches) else None,
    "false_positive_mask_area_pct": float(per_image["FPAreaPct"].mean()),
    "negative_false_positive_mask_area_pct": float(background["FPAreaPct"].mean()) if len(background) else None,
    "missed_overflow_rate_pct": divide(100 * fn, tp + fn),
    "background_images": len(background),
    "background_images_with_false_alarms": int(background["FP"].gt(0).sum()),
    "background_false_alarm_rate_pct": divide(100 * int(background["FP"].gt(0).sum()), len(background)),
    "deployment_latency_ms": None
}
per_class = ap_table.drop(index="All Classes").join(errors.per_class)
per_class["F1"] = 2 * per_class["TP"] / (2 * per_class["TP"] + per_class["FP"] + per_class["FN"])
per_video = per_image.groupby("Video").agg(Images=("Image", "size"), Objects=("Objects", "sum"), TP=("TP", "sum"), FP=("FP", "sum"), FN=("FN", "sum"))

per_video["Precision"] = per_video["TP"] / (per_video["TP"] + per_video["FP"]).replace(0, np.nan)
per_video["Recall"] = per_video["TP"] / (per_video["TP"] + per_video["FN"]).replace(0, np.nan)

for filename, table in (("per_class.csv", per_class), ("per_image.csv", per_image), ("per_video.csv", per_video)):
    table.to_csv(REPORT / filename)
    
boundary_matches.to_csv(REPORT / "boundary_matches.csv", index=False)
pd.DataFrame(metrics.items(), columns=["Metric", "Value"]).to_csv(REPORT / "metrics.csv", index=False)

display(ap_table.round(4))
display(pd.DataFrame(metrics.items(), columns=["Metric", "Value"]).round(4))
display(per_video.round(4))

,Instances,AP@0.50,AP@[.50:.95],AR@[.50:.95]
Class,,,,
Overflow,846,0.515,0.2737,0.4897
All Classes,846,0.515,0.2737,0.4897


,Metric,Value
0,mask_AP50,0.5150
1,mask_mAP50_95,0.2737
2,mask_AR50_95,0.4897
3,TP,307.0000
4,FP,137.0000
5,FN,539.0000
6,precision,0.6914
7,recall,0.3629
8,F1,0.4760
9,overall_foreground_IoU,0.5022


,Images,Objects,TP,FP,FN,Precision,Recall
Video,,,,,,,
0020,600,505,172,23,333,0.8821,0.3406
0031,170,120,56,48,64,0.5385,0.4667
0035,50,0,0,54,0,0.0000,NaN
0041,120,78,32,5,46,0.8649,0.4103
0062,100,86,1,3,85,0.2500,0.0116
0070,70,57,46,4,11,0.9200,0.8070


In [14]:
examples = per_image[per_image["Errors"].gt(0)].sort_values(["Errors", "FN", "FP", "Image"], ascending=[False, False, False, True]).drop_duplicates("Video").head(4)
selected_rows = []

for image_id, row in examples.iterrows():
    info = gt.imgs[image_id]
    image = cv2.cvtColor(cv2.imread(str(DATA / "test/images" / Path(info["file_name"]).name)), cv2.COLOR_BGR2RGB)
    
    truth = union_mask([gt.annToRLE(ann) for ann in gt.imgToAnns[image_id]], info["height"], info["width"])
    predicted = union_mask(operating[image_id], info["height"], info["width"])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, mask, title, color in zip(axes, (truth, predicted), ("Ground truth", f"S2 fused: confidence > {CONF}"), ((0.1, 0.9, 0.1), (1.0, 0.25, 0.1))):
        ax.imshow(image)
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[..., :3] = color
        overlay[..., 3] = mask * 0.45
        ax.imshow(overlay)
        ax.set_title(title)
        ax.axis("off")
        
    fig.suptitle(f"{row['Image']} | TP={row['TP']}, FP={row['FP']}, FN={row['FN']}")
    fig.tight_layout()
    filename = f"failure_{image_id}.jpg"
    fig.savefig(REPORT / filename, dpi=130, bbox_inches="tight")
    plt.close(fig)
    selected_rows.append({"Image ID": image_id, "Image": row["Image"], "Figure": filename})
    
pd.DataFrame(selected_rows, columns=["Image ID", "Image", "Figure"]).to_csv(REPORT / "failure_examples.csv", index=False)
record = {"status": "complete", "completed_utc": datetime.now(timezone.utc).isoformat(),
          "selection_id": selection_id, "run": RUN_NAME, "system": "S2_fused", "split": "test",
          "images": len(images), "instances": len(gt.anns), "metrics": metrics,
          "protocol": {"AP_evaluator": "pycocotools.COCOeval", "iou_type": "segm",
                       "AP_confidence_min": AP_CONF, "maxDets": [1, 10, 100],
                       "IoUs": "0.50:0.05:0.95", "operating_confidence": CONF,
                       "confidence_rule": ">", "match_mask_IoU": 0.5,
                       "matching": "COCO class-aware, score-ordered",
                       "pixel_metrics": "Pooled foreground pixels; empty/empty adds zero",
                       "boundary": "Mean matched-pair Boundary IoU, width=2% image diagonal",
                       "FP_area": "Mean per-image predicted foreground outside GT, as % of image area",
                       "latency": "Deferred to deployment benchmark; AP collection timings are not production latency"},
          "inference_environment": prediction_metadata["environment"], "evaluation_environment": environment()}

write_json(REPORT / "metrics.json", record)
summary = pd.DataFrame([
    {"System": "S0", "Split": "val", "Mask AP50": 0.26595271065932974, "Mask AP50-95": 0.14782122443828255},
    {"System": "S1", "Split": "val", "Mask AP50": 0.5299510239497677, "Mask AP50-95": 0.2775110925009392},
    {"System": "S2 fused", "Split": "val", "Mask AP50": selection["validation"]["mask_AP50"],
     "Mask AP50-95": selection["validation"]["mask_mAP50_95"]},
    {"System": "S2 fused", "Split": "test", "Mask AP50": metrics["mask_AP50"], "Mask AP50-95": metrics["mask_mAP50_95"]}
])

summary.to_csv(REPORT / "experiment_summary.csv", index=False)
metric_lines = "\n".join(f"| {name} | {value:.4f} |" for name, value in metrics.items() if value is not None)
gallery_links = "\n".join(f"- [{row['Image']}]({row['Figure']})" for row in selected_rows) or "No selected failures."
readme = f"""# {RUN_NAME} — frozen final test

System: D1 container proposals + frozen S1 full image + S2 ROI, fused at original image resolution.
S2 checkpoint: epoch 1. Evaluation image size: {IMGSZ}. Images: {len(images)}; instances: {len(gt.anns)}.
Operating confidence: **> {CONF}**, chosen on validation before test. No test tuning.

| Metric | Value |
| --- | --- |
{metric_lines}

## Protocol

Canonical COCO mask AP uses confidence 0.001, IoU 0.50:0.05:0.95, maxDets=[1,10,100].
P/R/F1 and pixel metrics use confidence > {CONF}; instance matching uses mask IoU >= 0.50.
Boundary IoU covers matched pairs only. Background false alarm rate counts negative images with at least one FP.
Foreground IoU/Dice pool pixels across the split. AP is independent of the chosen operating threshold.

ROI proposals come from D1 predictions, with the frozen crop policy and no GT proposals or test jitter.
No-container images keep the S1 branch. S1 precedes S2 on score ties in mask-IoU NMS.
S0/S1 used native checkpoint selection; S2 used canonical fused AP. Results represent one run.

## Saved evidence

- [Frozen system and checkpoint hashes](selected_system.json)
- [Metrics and software environments](metrics.json)
- [Per-class results](per_class.csv), [per-video results](per_video.csv), [per-image results](per_image.csv)
- [Experiment summary](experiment_summary.csv): validation comparisons and final test are separate rows.
- [Predictions](predictions.json), [original test ground truth](coco_ground_truth.json), [prediction metadata](prediction_metadata.json)
- [AP collection timings](ap_collection_times.csv): includes initialization; **not deployment latency**.
- Frozen evaluator, fusion helper, ROI policy and video manifest are stored beside this report.

## Selected errors

{gallery_links}

These examples document limitations; test results do not change this frozen configuration.
Next phase: package all three checkpoints, reproduce the pipeline, then benchmark and deploy.
S3 is deferred.
"""

(REPORT / "README.md").write_text(readme, encoding="utf-8")
write_json(REPORT / "evaluation_config.json", record)

display(summary.round(4))
print("Final test report saved:", REPORT)

,System,Split,Mask AP50,Mask AP50-95
0,S0,val,0.2660,0.1478
1,S1,val,0.5300,0.2775
2,S2 fused,val,0.6649,0.3650
3,S2 fused,test,0.5150,0.2737


Final test report saved: /content/drive/MyDrive/WasteScope/reports/segmentation/S2_yolo26n_seg_roi_960/test_best_960_fused_conf0.2
